In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

c:\Users\ASUS\Desktop\NLP-News-Intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "t5-small"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("T5-small loaded successfully!")

c:\Users\ASUS\Desktop\NLP-News-Intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ASUS\Desktop\NLP-News-Intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWa

T5-small loaded successfully!


In [5]:
from datasets import load_dataset

summarization_dataset = load_dataset(
    "phoenix60064/cnn_dailymail",
    "3.0.0"
)

print(summarization_dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [6]:
sample = summarization_dataset["test"][0]

article = sample["article"]
reference_summary = sample["highlights"]

print("ARTICLE:")
print(article)

print("\nREFERENCE SUMMARY:")
print(reference_summary)

ARTICLE:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday's cer

In [7]:
input_text = "summarize: " + article

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

In [8]:
summary_ids = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=100,
    min_length=20,
    num_beams=4,
    early_stopping=True
)

In [9]:
generated_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:")
print(generated_summary)

GENERATED SUMMARY:
the palestinians signed the ICC's founding Rome Statute in January. the ICC also accepted its jurisdiction over alleged crimes committed in the occupied territories. the ICC opened a preliminary examination into the situation in Palestinian territories.


In [10]:
generated_summary

"the palestinians signed the ICC's founding Rome Statute in January. the ICC also accepted its jurisdiction over alleged crimes committed in the occupied territories. the ICC opened a preliminary examination into the situation in Palestinian territories."

In [11]:
reference_summary

'Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .\nIsrael and the United States opposed the move, which could open the door to war crimes investigations against Israelis .'

In [12]:
from rouge_score import rouge_scorer

In [13]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

In [14]:
scores = scorer.score(
    reference_summary,
    generated_summary
)

print(scores)

{'rouge1': Score(precision=0.3783783783783784, recall=0.4117647058823529, fmeasure=0.39436619718309857), 'rouge2': Score(precision=0.2222222222222222, recall=0.24242424242424243, fmeasure=0.2318840579710145), 'rougeL': Score(precision=0.32432432432432434, recall=0.35294117647058826, fmeasure=0.3380281690140845)}


In [15]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [16]:
from src.summarization.summarizer import summarize_article

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 321.12it/s]


In [17]:
sample = summarization_dataset["test"][0]

article = sample["article"]
reference_summary = sample["highlights"]

In [18]:
generated_summary = summarize_article(article)

In [19]:
print("REFERENCE SUMMARY:")
print(reference_summary)

print("\nGENERATED SUMMARY:")
print(generated_summary)

REFERENCE SUMMARY:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

GENERATED SUMMARY:
the palestinians signed the ICC's founding Rome Statute in January. the ICC also accepted its jurisdiction over alleged crimes committed in the occupied territories. the ICC opened a preliminary examination into the situation in Palestinian territories.


In [20]:
my_article = """
Apple announced a new artificial intelligence system designed
to help users write, summarize documents and interact with
applications. The company said the technology will be integrated
into several of its products and services.
"""

In [21]:
summary = summarize_article(my_article)

print("SUMMARY:")
print(summary)

SUMMARY:
new artificial intelligence system will help users write, summarize documents and interact with applications. the technology will be integrated into several of its products and services.


In [22]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

print("ROUGE scorer ready!")

ROUGE scorer ready!


In [23]:
scores = scorer.score(
    reference_summary,
    generated_summary
)

print("ROUGE-1:", scores["rouge1"])
print("ROUGE-2:", scores["rouge2"])
print("ROUGE-L:", scores["rougeL"])

ROUGE-1: Score(precision=0.3783783783783784, recall=0.4117647058823529, fmeasure=0.39436619718309857)
ROUGE-2: Score(precision=0.2222222222222222, recall=0.24242424242424243, fmeasure=0.2318840579710145)
ROUGE-L: Score(precision=0.32432432432432434, recall=0.35294117647058826, fmeasure=0.3380281690140845)


In [24]:
import numpy as np

num_samples = 100

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for i in range(num_samples):
    sample = summarization_dataset["test"][i]

    article = sample["article"]
    reference = sample["highlights"]

    generated = summarize_article(article)

    scores = scorer.score(reference, generated)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{num_samples}")

print("\nAverage ROUGE Scores")
print("--------------------")
print(f"ROUGE-1: {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2: {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L: {np.mean(rougeL_scores):.4f}")

Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100

Average ROUGE Scores
--------------------
ROUGE-1: 0.3324
ROUGE-2: 0.1361
ROUGE-L: 0.2478


In [25]:
import pandas as pd
from pathlib import Path

rouge_results = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "F1_Score": [
        np.mean(rouge1_scores),
        np.mean(rouge2_scores),
        np.mean(rougeL_scores)
    ]
})

output_path = Path("../results/metrics/t5_small_rouge_scores.csv")

rouge_results.to_csv(output_path, index=False)

print("ROUGE results saved to:")
print(output_path)

print("\nSaved Results:")
print(rouge_results)

ROUGE results saved to:
..\results\metrics\t5_small_rouge_scores.csv

Saved Results:
    Metric  F1_Score
0  ROUGE-1  0.332399
1  ROUGE-2  0.136135
2  ROUGE-L  0.247807


In [26]:
import json
import torch

from src.pytorch.model import NewsClassifier

# Load vocabulary
with open("../data/processed/vocab.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)

# Create model
model = NewsClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=4,
    dropout=0.3
)

# Load trained weights
model.load_state_dict(
    torch.load(
        "../models/pytorch/best_news_classifier.pt",
        map_location="cpu"
    )
)

model.eval()

print("Vocabulary loaded:", len(vocab))
print("PyTorch model loaded successfully!")

Vocabulary loaded: 20000
PyTorch model loaded successfully!


In [27]:
from src.utils.helpers import analyze_news

news_article = """
Apple announced a new artificial intelligence system designed
to help users write, summarize documents and interact with
applications. The company said the technology will be integrated
into several of its products and services.
"""

result = analyze_news(
    news_article,
    model,
    vocab
)

print("CATEGORY:", result["category"])
print("CONFIDENCE:", f"{result['confidence']:.2%}")
print("SUMMARY:", result["summary"])

CATEGORY: Sci/Tech
CONFIDENCE: 99.20%
SUMMARY: new artificial intelligence system will help users write, summarize documents and interact with applications. the technology will be integrated into several of its products and services.


In [28]:
test_articles = {
    "Sports": """
    The national football team secured a dramatic victory in the
    international championship final after scoring two goals in
    the second half. The team's strong defensive performance helped
    them maintain their lead until the final whistle.
    """,

    "Business": """
    The company reported strong financial results for the latest
    quarter, with revenue increasing significantly compared with
    the previous year. Executives said continued investment and
    growing customer demand contributed to the improved performance.
    """,

    "World": """
    Government leaders from several countries met today to discuss
    international security and cooperation. The meeting focused on
    regional stability, economic development and plans to strengthen
    diplomatic relations.
    """,

    "Sci/Tech": """
    Researchers have developed a new artificial intelligence system
    that can analyze large amounts of information and help users
    complete complex tasks. The technology is expected to be used
    in several applications in the future.
    """
}

for expected_category, article in test_articles.items():

    result = analyze_news(
        article,
        model,
        vocab
    )

    print("=" * 60)
    print("EXPECTED:", expected_category)
    print("PREDICTED:", result["category"])
    print("CONFIDENCE:", f"{result['confidence']:.2%}")
    print("SUMMARY:", result["summary"])

EXPECTED: Sports
PREDICTED: Sports
CONFIDENCE: 99.13%
SUMMARY: the national football team secured a dramatic victory in the international championship final. the team's defensive performance helped them maintain their lead until the final whistle.
EXPECTED: Business
PREDICTED: Business
CONFIDENCE: 95.82%
SUMMARY: the company reported strong financial results for the latest quarter. revenue increased significantly compared with the previous year.
EXPECTED: World
PREDICTED: World
CONFIDENCE: 96.62%
SUMMARY: government leaders from several countries met today to discuss international security and cooperation. meeting focused on regional stability, economic development and plans to strengthen diplomatic relations.
EXPECTED: Sci/Tech
PREDICTED: Sci/Tech
CONFIDENCE: 98.23%
SUMMARY: researchers have developed a new artificial intelligence system that can analyze large amounts of information. the technology is expected to be used in several applications in the future.
